# Transformer Based Multimodal Trajectory Prediction on the Waymo Open Motion Dataset

Runs the whole pipeline with the code in this repo- data prep, anchors, training, evaluation and plots.
By default it skips training and evaluates the checkpoint in `checkpoints/trajectory_transformer_best.pt`
on the test split.
This notebook is designed to be used with Google Colab. Prepped data and training runs live on Google Drive. Raw Waymo shards are read from the `waymo_open_dataset_motion_v_1_2_0` bucket.

## Setup

In [ ]:
# Get the code onto the runtime: use the .py files next to this notebook if they are there
# (local runtime, or uploaded to Colab), otherwise clone the repo.
import os

if not os.path.exists("model.py"):
    !git clone -q https://github.com/RajPShinde/Transformer-Based-Multimodal-Trajectory-Prediction.git
    %cd Transformer-Based-Multimodal-Trajectory-Prediction
!pip install -q torch==2.9.1 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch

import config
from anchors import build_anchors
from metrics import evaluate, ranking_gap_table
from model import load_checkpoint
from prep import describe_splits, load_splits, prep_done, run_prep
from train import smoke_test, train
from visualize import plot_test_predictions

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count())
print(f"PyTorch {torch.__version__}, device: {DEVICE}, CPU threads: {torch.get_num_threads()}")

## Options

All model, loss and training settings are in `config.py`.

In [ ]:
TRAIN = False          # True: fit anchors and train from scratch (A100, 10 h budget); False: evaluate CHECKPOINT
RESUME_RUN_DIR = None  # e.g. f"{config.DRIVE_DIR}/run_20261004_2029" to continue an interrupted training run
CHECKPOINT = f"{config.DRIVE_DIR}/run_20261004_2029/trajectory_transformer_best.pt"   # the run behind the README results
FORCE_PREP = False     # prep even on a GPU runtime

## Google Drive and Waymo access

Drive holds the prepped data and training runs. `gcloud auth login` is only needed to download raw shards.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Sign in to Google Cloud (needed to download raw shards). Two cells because gcloud reads the verification
# code from stdin, which VS Code's Colab extension can't type into; the code goes in through input() instead.
# This cell starts gcloud and prints the sign-in link; the next one takes the code.
import subprocess

p = subprocess.Popen(["gcloud", "auth", "login", "--no-launch-browser"], stdin=subprocess.PIPE,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:                 # print until the sign-in link appears
    print(line, end="")
    if "accounts.google.com" in line:
        break

In [ ]:
# Open the link above, sign in, copy the code and paste it here.
code = input("Paste the verification code: ")
print(p.communicate(code + "\n")[0])

## Data

Tensorizes the raw shards to `config.PREP_OUT_DIR` on Drive. Later runs load the compact tensors. Evaluation only needs the test split.

In [ ]:
if prep_done() and not FORCE_PREP:
    train_scenes, val_scenes, test_scenes = load_splits(("train", "val", "test") if TRAIN else ("test",))
    describe_splits(train_scenes, val_scenes)
elif torch.cuda.is_available() and not FORCE_PREP:
    print("No prepped data found and this is a GPU runtime. Run prep on a CPU runtime first, "
          "then come back to the GPU to load. Set FORCE_PREP = True to override.")
else:
    run_prep()

## Anchors

Each ground truth future gets a maneuver group from its geometry. Anchors are fit by k-means inside each group. The alchors also get saved in the checkpoint.
The smoke test checks the GPU slot-assignment rule against numpy and runs a short training burst.

In [ ]:
if TRAIN:
    anchors = build_anchors(train_scenes, val_scenes, DEVICE)
    smoke_test(train_scenes, anchors, DEVICE)

## Train

In [ ]:
if TRAIN:
    CHECKPOINT = train(train_scenes, val_scenes, anchors, DEVICE,
                       resume_run_dir=RESUME_RUN_DIR, n_test=len(test_scenes))

## Load the checkpoint

In [ ]:
model, ckpt, _ = load_checkpoint(CHECKPOINT, DEVICE,
                                 session_anchors=anchors["grouped_anchor_paths"] if TRAIN else None)

## Evaluate

Top-6 of 32 modes on the held-out test split: minADE / minFDE / miss rate (2 m at 8 s), calibration, off road
rate, per agent type, Waymo-style metrics, and the gap to perfect ranking.

In [ ]:
if TRAIN:
    evaluate(model, val_scenes, "VAL (used for selection)")
evaluate(model, test_scenes, "TEST (held out)");

In [ ]:
ranking_gap_table(model, test_scenes);

## Prediction plots

Turn-like, random and worst-case test scenes: top-1, scored top-6 and all 32 modes. Saved to `output/`.

In [ ]:
plot_test_predictions(model, save_dir="output")